In [0]:
# %sql

# CREATE SCHEMA IF NOT EXISTS ECOMMERCE.SILVER;

In [0]:
%sql
SELECT * FROM ecommerce.bronze.orders limit 10;

%md
## Loading Libraries for future usage

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col


%md
# Loading data from Bronze layer

## Business Rule 1: Drop Duplicates 

We load the data from the bronze layer table and we drop duplicates (requirement as a part of customer business rule 1- drop duplicates )

In [0]:
orders_df = (
    spark.read.table("ecommerce.bronze.orders")
    .dropDuplicates()
    .withColumn("order_id", col("order_id").cast("bigint"))
    .withColumn("customer_id", col("customer_id").cast("double").cast("int"))
    .withColumn("order_date", col("order_date").cast("timestamp"))
    .withColumn("created_at", col("created_at").cast("timestamp"))
    .withColumn("updated_at", col("updated_at").cast("timestamp"))
    .withColumn("total_amount", col("total_amount").cast("decimal(18,2)"))
)

In [0]:
orders_df.printSchema()

## Business Rule 2: Validate customer IDs.

We can clearly see that the cusomer ids are floating type and we need to convert them to Integer valued

In [0]:
orders_df = orders_df.withColumn("customer_id", col("customer_id").try_cast("int"))
orders_df = orders_df.filter(col("customer_id").isNotNull())

## Business Rule 3: Standardize status.
We have available statuses as PLACED, CONFIRMED, SHIPPED, DELIVERED, CANCELLED, RETURNED

In [0]:
valid_statuses = [
    "PLACED",
    "CONFIRMED",
    "SHIPPED",
    "DELIVERED",
    "CANCELLED",
    "RETURNED",
]

orders_df = orders_df.withColumn(
    "order_status", upper(trim(col("order_status")))
).filter(col("order_status").isin(valid_statuses))

## Business Rule 4: Validate Order dates

In [0]:
orders_df = orders_df.withColumn(
    "valid_order_date", when(col("order_date") < "2026-09-10", 1).otherwise(0)
)

orders_df = orders_df.filter(col("valid_order_date") == 1)

orders_df = orders_df.withColumn("order_date_updated", to_date(col("order_date")))

## Business Rule 5: Reject or quarantine negative order amounts.

In [0]:
orders_df = orders_df.filter(col('total_amount')>0)

## Business Rule 6: Identify cancelled orders.

In [0]:
orders_df = orders_df.withColumn('Cancelled_orders', when(col('order_status')=='CANCELLED',1).otherwise(0))

## Business Rule 7: Identify Returned orders

In [0]:
orders_df = orders_df.withColumn('Returned_orders', when(col('order_status')=='RETURNED',1).otherwise(0))

## Business Rule 8: Identify Completed orders

In [0]:
orders_df = orders_df.withColumn('Completed_orders', when(col('order_status')=='DELIVERED',1).otherwise(0))

In [0]:
orders_df.write.format("delta").mode("append").saveAsTable(
    "ecommerce.silver.orders"
)

In [0]:
%sql
SELECT * FROM ECOMMERCE.SILVER.ORDERS LIMIT 100;